# Challenge 2: entrenamiento y selección del modelo

En esta etapa debes construir pipelines reproducibles, comparar modelos y seleccionar un campeón sin utilizar el conjunto de test.

**Entradas:** `train.csv` y `data_contract.json`  
**Salidas:** `champion_model.joblib`, `training_metadata.json` y `cv_results.csv`

## Reglas

- El preprocesamiento debe estar dentro de cada `Pipeline`.
- La selección debe hacerse con validación cruzada.
- No cargues `test.csv` en este notebook.

In [1]:
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

In [2]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
REPORTS_DIR = PROJECT_DIR / "reports"

for directory in (PROCESSED_DATA_DIR, ARTIFACTS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

## 1. Carga de artefactos

In [3]:
train_df = pd.read_csv(PROCESSED_DATA_DIR / "train.csv")
contract = json.loads((ARTIFACTS_DIR / "data_contract.json").read_text(encoding="utf-8"))

X_train = train_df[contract["features"]]
y_train = train_df[contract["target"]]

print(X_train.shape, y_train.shape)

(142, 13) (142,)


## 2. Pipeline de procesamiento

In [4]:
feature_columns = contract["features"]

numeric_for_tree = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
])

numeric_scaled = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

preprocessor_tree = ColumnTransformer([
    ("numeric", numeric_for_tree, feature_columns),
])

preprocessor_scaled = ColumnTransformer([
    ("numeric", numeric_scaled, feature_columns),
])

preprocessors = {
    "tree": preprocessor_tree,
    "scaled": preprocessor_scaled,
}

**Pregunta:** ¿por qué el `StandardScaler` debe estar dentro del pipeline y no ajustarse antes de la validación cruzada?

> Si el `StandardScaler` se ajusta (`fit`) sobre todo `X_train` antes de hacer la validación cruzada, la media y la desviación estándar usadas para escalar ya habrían "visto" los datos de cada fold de validación, aunque esos datos no se usen para entrenar el modelo en esa iteración. Eso es una fuga de información (*data leakage*): el fold de validación deja de ser realmente independiente, porque su información estadística ya influyó en la transformación. Poniendo el `StandardScaler` dentro del `Pipeline`, `cross_validate`/`GridSearchCV` lo ajustan de nuevo en cada fold usando solo los datos de entrenamiento de ese fold, simulando correctamente lo que pasará en producción: el scaler se ajusta una sola vez con los datos de entrenamiento y luego se aplica (sin volver a ajustar) a datos nuevos.

## 3. Modelos candidatos

In [5]:
models = {
    "decision_tree": Pipeline([
        ("preprocessing", clone(preprocessors["tree"])),
        ("classifier", DecisionTreeClassifier(random_state=RANDOM_STATE)),
    ]),
    "logistic_regression": Pipeline([
        ("preprocessing", clone(preprocessors["scaled"])),
        ("classifier", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    ]),
    "knn": Pipeline([
        ("preprocessing", clone(preprocessors["scaled"])),
        ("classifier", KNeighborsClassifier()),
    ]),
}

models

{'decision_tree': Pipeline(steps=[('preprocessing',
                  ColumnTransformer(transformers=[('numeric',
                                                   Pipeline(steps=[('imputer',
                                                                    SimpleImputer(strategy='median'))]),
                                                   ['alcohol', 'malic_acid',
                                                    'ash', 'alcalinity_of_ash',
                                                    'magnesium', 'total_phenols',
                                                    'flavanoids',
                                                    'nonflavanoid_phenols',
                                                    'proanthocyanins',
                                                    'color_intensity', 'hue',
                                                    'od280/od315_of_diluted_wines',
                                                    'proline'])])),
                 ('c

## 4. Comparación inicial con validación cruzada

In [6]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

rows = []
for name, pipeline in models.items():
    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring="accuracy",
        return_train_score=True,
        n_jobs=-1,
    )
    rows.append({
        "model": name,
        "train_accuracy_mean": scores["train_score"].mean(),
        "val_accuracy_mean": scores["test_score"].mean(),
        "val_accuracy_std": scores["test_score"].std(),
    })

initial_comparison = pd.DataFrame(rows).sort_values("val_accuracy_mean", ascending=False).reset_index(drop=True)
initial_comparison

,model,train_accuracy_mean,val_accuracy_mean,val_accuracy_std
0,logistic_regression,1.0000,0.979064,0.027713
1,knn,0.9771,0.950739,0.035274
2,decision_tree,1.0000,0.902217,0.058617


## 5. Optimización de hiperparámetros

In [7]:
param_grids = {
    "decision_tree": {
        "classifier__max_depth": [2, 3, 4, 5, None],
        "classifier__min_samples_leaf": [1, 2, 4],
    },
    "logistic_regression": {
        "classifier__C": [0.01, 0.1, 1.0, 10.0],
    },
    "knn": {
        "classifier__n_neighbors": [3, 5, 7, 9, 11],
        "classifier__weights": ["uniform", "distance"],
    },
}

searches = {}
tuned_rows = []

for name, pipeline in models.items():
    search = GridSearchCV(
        pipeline,
        param_grids[name],
        scoring="accuracy",
        cv=cv,
        n_jobs=-1,
    )
    search.fit(X_train, y_train)
    searches[name] = search
    tuned_rows.append({
        "model": name,
        "best_params": search.best_params_,
        "best_cv_accuracy": search.best_score_,
        "cv_accuracy_std": search.cv_results_["std_test_score"][search.best_index_],
    })

tuned_comparison = pd.DataFrame(tuned_rows).sort_values("best_cv_accuracy", ascending=False).reset_index(drop=True)
tuned_comparison

,model,best_params,best_cv_accuracy,cv_accuracy_std
0,logistic_regression,{'classifier__C': 1.0},0.979064,0.027713
1,knn,"{'classifier__n_neighbors': 7, 'classifier__we...",0.957882,0.040520
2,decision_tree,"{'classifier__max_depth': 5, 'classifier__min_...",0.902463,0.076225


## 6. Selección y persistencia del campeón

In [8]:
champion_name = tuned_comparison.loc[0, "model"]
champion = searches[champion_name].best_estimator_

joblib.dump(champion, ARTIFACTS_DIR / "champion_model.joblib")

training_metadata = {
    "champion": champion_name,
    "best_params": searches[champion_name].best_params_,
    "cv_accuracy_mean": float(searches[champion_name].best_score_),
    "cv_accuracy_std": float(
        searches[champion_name].cv_results_["std_test_score"][searches[champion_name].best_index_]
    ),
    "candidates": tuned_comparison.to_dict(orient="records"),
}
(ARTIFACTS_DIR / "training_metadata.json").write_text(
    json.dumps(training_metadata, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

cv_results_frames = []
for name, search in searches.items():
    frame = pd.DataFrame(search.cv_results_)
    frame["model"] = name
    cv_results_frames.append(frame)

cv_results = pd.concat(cv_results_frames, ignore_index=True)
cv_results.to_csv(REPORTS_DIR / "cv_results.csv", index=False)

print("Campeón:", champion_name)
training_metadata

Campeón: logistic_regression


{'champion': 'logistic_regression',
 'best_params': {'classifier__C': 1.0},
 'cv_accuracy_mean': 0.9790640394088669,
 'cv_accuracy_std': 0.027713464392187602,
 'candidates': [{'model': 'logistic_regression',
   'best_params': {'classifier__C': 1.0},
   'best_cv_accuracy': 0.9790640394088669,
   'cv_accuracy_std': 0.027713464392187602},
  {'model': 'knn',
   'best_params': {'classifier__n_neighbors': 7,
    'classifier__weights': 'uniform'},
   'best_cv_accuracy': 0.9578817733990148,
   'cv_accuracy_std': 0.040520048714638005},
  {'model': 'decision_tree',
   'best_params': {'classifier__max_depth': 5,
    'classifier__min_samples_leaf': 2},
   'best_cv_accuracy': 0.9024630541871922,
   'cv_accuracy_std': 0.07622506080471962}]}

In [9]:
assert (ARTIFACTS_DIR / "champion_model.joblib").exists()
assert (ARTIFACTS_DIR / "training_metadata.json").exists()
assert (REPORTS_DIR / "cv_results.csv").exists()
print("Challenge 2 completado.")

Challenge 2 completado.


**Justificación de la selección:**

> El campeón es `logistic_regression` (`C=1.0`), con accuracy promedio en CV de ≈0.979 (±0.028), claramente por encima de KNN (≈0.958, `n_neighbors=7`) y del árbol de decisión (≈0.902, `max_depth=5`). Además de tener el mejor desempeño, es el modelo con menor variabilidad entre folds después de KNN, lo que indica que generaliza de forma consistente y no depende de una partición particular de los datos. Su complejidad es baja (un modelo lineal con 13 coeficientes por clase) y es el más interpretable de los tres: los coeficientes indican directamente el efecto de cada variable química sobre la probabilidad de cada clase, a diferencia de KNN (no paramétrico, sin coeficientes interpretables) o del árbol (interpretable como reglas, pero con mayor varianza y mayor riesgo de sobreajuste al dataset de entrenamiento, visible en su desviación estándar más alta, 0.076). Dado que el dataset es pequeño (142 filas en train) y las clases se separan razonablemente bien de forma lineal (ver hallazgos del EDA en el challenge 1), un modelo lineal regularizado es una elección razonable y poco propensa a sobreajuste.